# Bài toán 2 — Gợi ý kỹ năng tiếp theo từ role, cấp bậc kinh nghiệm và kỹ năng đã biết

Với role mục tiêu và những kỹ năng người học đã có, gợi ý kỹ năng phù hợp để học tiếp. Bản gốc khai phá luật theo từng role, tương đương `{ROLE, known skill(s)} → next skill`. Bản này thêm **cấp bậc kinh nghiệm** (`Intern`, `Fresher`, `Junior`, `Senior`, suy ra từ `experience_required`) vào vế trái, tức `{ROLE, EXP, known skill(s)} → next skill`, rồi so sánh với luật gốc và với baseline tần suất.

## 1. Định nghĩa bài toán và tránh rò rỉ dữ liệu

Đây là bài toán **khôi phục kỹ năng bị ẩn**, không phải dự đoán thứ tự học theo thời gian. Với mỗi tin tuyển dụng ở tập test, ẩn có kiểm soát 20% kỹ năng đã trích xuất; các kỹ năng còn lại cùng role và **cấp bậc kinh nghiệm của tin** tạo thành ngữ cảnh truy vấn, còn kỹ năng bị ẩn là mục tiêu. Cấp bậc là thông tin biết trước của tin tuyển dụng nên không làm lộ mục tiêu. Luật chỉ được học từ các fold train. Baseline nhận cùng ngữ cảnh role (và cấp bậc ở biến thể + EXP) và loại các kỹ năng đã biết khỏi danh sách gợi ý, nhưng không dùng ngữ cảnh skill. Loại `Other` và tin không có skill khỏi dữ liệu; tin chỉ có một skill vẫn được dùng trong train nhưng không dùng làm truy vấn test vì không thể vừa có ngữ cảnh vừa có mục tiêu.

In [ ]:
import ast, json, math, time, tracemalloc
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from scipy.stats import wilcoxon
from sklearn.model_selection import RepeatedStratifiedKFold

DATA_DIR = Path("../data/processed")
RESULT_DIR = Path("../result/problem_2")
FIGURES_DIR = RESULT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE, N_SPLITS, N_REPEATS = 42, 5, 3   # 5 fold x 3 lần lặp = 15 lần đánh giá (đặt N_REPEATS=1 để chạy nhanh)
MIN_SUPPORT_COUNT, MIN_CONFIDENCE, MIN_LIFT = 5, 0.20, 1.20
MAX_ITEMSET_LENGTH = 3  # per role: vế trái tối đa 2 item (skill hoặc 1 item EXP) -> 1 skill được gợi ý
SMOOTH_ALPHA, MIN_CELL_JOBS = 20, 10   # tham số baseline "Phổ biến theo role + EXP"
EXP_LEVELS = ["Intern", "Fresher", "Junior", "Senior"]
TOP_K_VALUES, MASK_RATIO = [5, 10], 0.20

def parse_skill_list(value):
    if isinstance(value, list): parsed = value
    elif isinstance(value, str):
        try: parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError): parsed = []
    else: parsed = []
    if not isinstance(parsed, list): return []
    return list(dict.fromkeys(x.strip() for x in parsed if isinstance(x, str) and x.strip()))

jobs_all = pd.read_csv(DATA_DIR / "processed_jobs.csv")
jobs_all["skills_final"] = jobs_all["skills_final"].apply(parse_skill_list)
if "exp_level" not in jobs_all:
    raise ValueError("Thiếu cột exp_level. Hãy chạy lại notebook EDA_and_Preprocessing trước.")
jobs_all["exp_level"] = jobs_all["exp_level"].fillna("Unknown")
role_mask = jobs_all["role_group"].fillna("Other").ne("Other")
jobs = jobs_all.loc[role_mask].reset_index(drop=True)
transactions = pd.read_csv(DATA_DIR / "transaction_matrix.csv").replace(
    {"True": True, "False": False}
).fillna(False).astype(bool).reset_index(drop=True)
if len(jobs) != len(transactions):
    raise ValueError("processed_jobs và transaction_matrix không khớp thứ tự dòng sau khi bỏ Other")
role_cols = [c for c in transactions if c.startswith("ROLE_")]
exp_cols = [c for c in transactions if c.startswith("EXP_")]
skill_cols = [c for c in transactions if not c.startswith(("ROLE_", "EXP_"))]
if not exp_cols:
    raise ValueError("Ma trận giao dịch chưa có cột EXP_*. Hãy chạy lại notebook EDA_and_Preprocessing.")
role_from_matrix = transactions[role_cols].idxmax(axis=1).str.replace("ROLE_", "", n=1, regex=False)
if not role_from_matrix.equals(jobs.role_group.reset_index(drop=True)):
    raise ValueError("Nhãn role không khớp thứ tự dòng của ma trận giao dịch")
exp_from_matrix = pd.Series("Unknown", index=transactions.index)
has_exp = transactions[exp_cols].any(axis=1)
exp_from_matrix[has_exp] = transactions.loc[has_exp, exp_cols].idxmax(axis=1).str.replace("EXP_", "", n=1, regex=False)
if not exp_from_matrix.equals(jobs["exp_level"].reset_index(drop=True)):
    raise ValueError("Nhãn exp_level không khớp ma trận giao dịch")
has_skills = jobs.skills_final.map(bool)
excluded_empty = int((~has_skills).sum())
jobs = jobs.loc[has_skills].reset_index(drop=True)
transactions = transactions.loc[has_skills].reset_index(drop=True)
query_mask = jobs.skills_final.map(lambda xs: len(set(xs) & set(skill_cols)) >= 2)
excluded_short = int((~query_mask).sum())
jobs["query_eligible"] = query_mask.to_numpy()
if jobs.role_group.value_counts().min() < N_SPLITS:
    raise ValueError("Một role có quá ít tin để chia phân tầng 5-fold")
print(f"Tin train có skill: {len(jobs)}; truy vấn test đủ điều kiện: {int(query_mask.sum())}; "
      f"loại do thiếu skill: {excluded_empty}; tin không đủ làm truy vấn: {excluded_short}; "
      f"số role: {jobs.role_group.nunique()}")
display(pd.crosstab(jobs.loc[jobs.query_eligible, "role_group"], jobs.loc[jobs.query_eligible, "exp_level"])
        .reindex(columns=EXP_LEVELS, fill_value=0).assign(Tổng=lambda d: d.sum(axis=1)))

## 2. Khai phá luật theo từng role

Khai phá riêng từng role tương đương với việc thêm role vào vế trái của mọi luật. Ví dụ, luật `SQL → Power BI` khai phá trong các tin Data Analyst tương đương `{Data Analyst, SQL} → Power BI`. Mỗi luật cần xuất hiện trong ít nhất 5 tin của role, có confidence ≥ 0.20 và lift ≥ 1.20. File xuất kèm số tin hỗ trợ vì luật theo ngữ cảnh có thể thưa.

**Biến thể + EXP:** trong ma trận của từng role, thêm các cột `EXP_*` (chỉ giữ cột đủ 5 tin). Vế trái tối đa hai item và có thể chứa một item `EXP_*`, ví dụ `{EXP_Senior, Java} → Spring` hoặc `{EXP_Intern} → Excel`. Khi gợi ý, ngữ cảnh truy vấn gồm skill đã biết **cộng** item cấp bậc của tin. Tập luật của biến thể + EXP chứa toàn bộ luật gốc, cộng thêm các luật có item EXP, nên chênh lệch giữa hai mô hình chỉ đến từ thông tin cấp bậc.

Baseline **Phổ biến theo role + EXP** xếp hạng skill theo tần suất trong ô (role, cấp bậc) có làm mượt về tần suất role: `(đếm_ô + α·p_role) / (n_ô + α)`; ô dưới `MIN_CELL_JOBS` tin thì dùng tần suất role. Đây là đối chứng cho biết lợi ích đến từ thông tin kinh nghiệm hay từ cơ chế luật.

In [ ]:
RULE_COLS = ["role_group", "known_skills", "next_skill", "support", "support_count",
             "confidence", "lift", "role_training_jobs"]

def build_rules(freq, n_rows):
    # mlxtend >= 0.23.4 yêu cầu num_itemsets; các bản cũ hơn không nhận tham số này.
    try:
        return association_rules(freq, num_itemsets=n_rows, metric="confidence", min_threshold=MIN_CONFIDENCE)
    except TypeError:
        return association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)

def mine_context_rules(train_jobs, train_matrix, method, use_exp=False):
    miner = apriori if method == "Apriori" else fpgrowth
    item_cols = skill_cols + (exp_cols if use_exp else [])
    rows = []
    for role in sorted(train_jobs.role_group.unique()):
        pos = np.flatnonzero(train_jobs.role_group.eq(role).to_numpy())
        role_matrix = train_matrix.iloc[pos][item_cols]
        counts = role_matrix.sum(axis=0)
        active = counts[counts >= MIN_SUPPORT_COUNT].index.tolist()
        if len(active) < 2 or len(role_matrix) < MIN_SUPPORT_COUNT:
            continue
        role_matrix = role_matrix[active]
        freq = miner(role_matrix, min_support=MIN_SUPPORT_COUNT / len(role_matrix),
                     use_colnames=True, max_len=MAX_ITEMSET_LENGTH)
        if freq.empty or not freq.itemsets.map(len).ge(2).any():
            continue
        rules = build_rules(freq, len(role_matrix))
        if rules.empty: continue
        rules = rules[
            rules.antecedents.map(lambda x: 1 <= len(x) <= 2)
            & rules.consequents.map(lambda x: len(x) == 1 and not str(next(iter(x))).startswith("EXP_"))
            & (rules.lift >= MIN_LIFT)
            & (rules.support * len(role_matrix) >= MIN_SUPPORT_COUNT - 1e-9)
        ]
        for r in rules.itertuples(index=False):
            rows.append({"role_group": role, "known_skills": frozenset(r.antecedents),
                "next_skill": str(next(iter(r.consequents))), "support": float(r.support),
                "support_count": int(round(r.support * len(role_matrix))),
                "confidence": float(r.confidence), "lift": float(r.lift),
                "role_training_jobs": len(role_matrix)})
    result = pd.DataFrame(rows, columns=RULE_COLS)
    if not result.empty:
        result = result.sort_values(["role_group", "confidence", "lift", "support_count"],
                                    ascending=[True, False, False, False]).reset_index(drop=True)
    return result

def rule_index(rules):
    index = {}
    for r in rules.itertuples(index=False):
        index.setdefault(r.role_group, []).append(
            (r.known_skills, r.next_skill, r.confidence, len(r.known_skills), r.lift, r.support_count))
    return index

def recommend_context(index, role, known, k):
    known = set(known); best = {}
    for antecedent, candidate, confidence, specificity, lift, count in index.get(role, []):
        if antecedent.issubset(known) and candidate not in known:
            score = (confidence, specificity, lift, count)
            if candidate not in best or score > best[candidate]: best[candidate] = score
    return [x for x, _ in sorted(best.items(), key=lambda item: item[1], reverse=True)[:k]]

def popularity_index(train_jobs, train_matrix):
    # Trả về (xếp hạng theo role, xếp hạng theo ô role+EXP có làm mượt và backoff).
    X = train_matrix[skill_cols].to_numpy(dtype=float)
    roles, exps = train_jobs.role_group.to_numpy(), train_jobs.exp_level.to_numpy()
    order = lambda scores: [skill_cols[i] for i in np.argsort(-scores, kind="stable")]
    role_rank, cell_rank = {}, {}
    for role in np.unique(roles):
        in_role = roles == role
        role_sum = X[in_role].sum(axis=0)
        role_rank[role] = order(role_sum)                      # đếm thô, giống baseline gốc
        prior = role_sum / in_role.sum()
        for exp in EXP_LEVELS:
            in_cell = in_role & (exps == exp)
            if in_cell.sum() >= MIN_CELL_JOBS:
                score = (X[in_cell].sum(axis=0) + SMOOTH_ALPHA * prior) / (in_cell.sum() + SMOOTH_ALPHA)
                cell_rank[(role, exp)] = order(score)
    return role_rank, cell_rank

def score_at_k(prediction, target, known, k):
    known, target = set(known), set(target) - set(known)
    prediction = [x for x in dict.fromkeys(prediction) if x not in known][:k]
    hits = [int(x in target) for x in prediction]
    dcg = sum(hit / math.log2(i + 2) for i, hit in enumerate(hits))
    idcg = sum(1 / math.log2(i + 2) for i in range(min(k, len(target))))
    return {"Precision@K": sum(hits) / k,
        "Recall@K": sum(hits) / len(target) if target else 0.0,
        "Hit@K": float(any(hits)), "NDCG@K": dcg / idcg if idcg else 0.0,
        "Recommendation coverage": len(prediction) / k}

def make_queries(test_jobs, fold):
    rng = np.random.default_rng(RANDOM_STATE + fold); result = []
    for i, row in test_jobs.iterrows():
        skills = sorted(set(row.skills_final) & set(skill_cols))
        n_hidden = min(len(skills) - 1, max(1, math.ceil(len(skills) * MASK_RATIO)))
        hidden_idx = set(rng.choice(len(skills), size=n_hidden, replace=False).tolist())
        result.append({"job_index": int(i), "role_group": row.role_group, "exp_level": row.exp_level,
            "known": {s for j, s in enumerate(skills) if j not in hidden_idx},
            "target": {s for j, s in enumerate(skills) if j in hidden_idx}})
    return result

## 3. Đánh giá khôi phục skill bị ẩn bằng 5-fold lặp lại

Với mỗi tin test, mô hình nhận role, (với biến thể + EXP) cấp bậc kinh nghiệm và các skill đã biết rồi dự đoán skill bị ẩn. `Precision@K`, `Recall@K`, `Hit@K` và `NDCG@K` đánh giá độ chính xác, độ bao phủ, tỷ lệ có gợi ý trúng và chất lượng thứ hạng. Phép chia được lặp `N_REPEATS` lần để có nhiều quan sát cho so sánh cặp; cần xem đồng thời chất lượng và độ bao phủ. Chi phí mô hình + EXP chỉ gồm lần khai phá có item EXP (không cộng lần khai phá luật gốc vì không dùng lại luật gốc).

In [ ]:
MODELS = ["Apriori", "FP-Growth", "Phổ biến theo role",
          "Apriori + EXP", "FP-Growth + EXP", "Phổ biến theo role + EXP"]
metric_names = ["Precision@K", "Recall@K", "Hit@K", "NDCG@K", "Recommendation coverage"]
MAX_K = max(TOP_K_VALUES)
splitter = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
fold_quality, fold_benchmarks, fold_role_quality, examples = [], [], [], []

def timed(fn):
    tracemalloc.start(); started = time.perf_counter()
    result = fn()
    runtime = time.perf_counter() - started
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return result, runtime, peak / 2**20

for fold, (tr, te) in enumerate(splitter.split(np.zeros(len(jobs)), jobs.role_group), 1):
    train_jobs = jobs.iloc[tr].reset_index(drop=True)
    test_all = jobs.iloc[te]
    test_jobs = test_all[test_all.query_eligible].reset_index(drop=True)
    train_matrix = transactions.iloc[tr].reset_index(drop=True)
    queries = make_queries(test_jobs, fold)
    predictors = {}
    for method in ["Apriori", "FP-Growth"]:
        for use_exp in [False, True]:
            name = method + (" + EXP" if use_exp else "")
            rules, runtime, memory = timed(lambda: mine_context_rules(train_jobs, train_matrix, method, use_exp))
            index = rule_index(rules)
            if use_exp:
                predictors[name] = lambda q, index=index: recommend_context(
                    index, q["role_group"],
                    q["known"] | ({f"EXP_{q['exp_level']}"} if q["exp_level"] != "Unknown" else set()), MAX_K)
            else:
                predictors[name] = lambda q, index=index: recommend_context(index, q["role_group"], q["known"], MAX_K)
            fold_benchmarks.append({"fold": fold, "Model": name, "Runtime (s)": runtime,
                "Memory Usage (MB)": memory, "Number of Rules": len(rules)})
    (pop_role, pop_cell), runtime, memory = timed(lambda: popularity_index(train_jobs, train_matrix))
    predictors["Phổ biến theo role"] = lambda q: [x for x in pop_role.get(q["role_group"], [])
                                                  if x not in q["known"]][:MAX_K]
    predictors["Phổ biến theo role + EXP"] = lambda q: [x for x in pop_cell.get(
        (q["role_group"], q["exp_level"]), pop_role.get(q["role_group"], [])) if x not in q["known"]][:MAX_K]
    for name in ["Phổ biến theo role", "Phổ biến theo role + EXP"]:
        fold_benchmarks.append({"fold": fold, "Model": name, "Runtime (s)": runtime,
            "Memory Usage (MB)": memory, "Number of Rules": np.nan})

    predictions = {name: [predictors[name](q) for q in queries] for name in MODELS}
    for name in MODELS:
        role_scores = {}
        fold_scores = {k: [] for k in TOP_K_VALUES}
        for q, prediction in zip(queries, predictions[name]):
            for k in TOP_K_VALUES:
                score = score_at_k(prediction, q["target"], q["known"], k)
                fold_scores[k].append(score)
                role_scores.setdefault((q["role_group"], k), []).append(score)
        for k, scores in fold_scores.items():
            fold_quality.append({"fold": fold, "Model": name, "K": k,
                **{m: float(np.mean([x[m] for x in scores])) for m in scores[0]},
                "query_count": len(queries)})
        for (role, k), scores in role_scores.items():
            fold_role_quality.append({"fold": fold, "Model": name, "role_group": role, "K": k,
                **{m: np.mean([x[m] for x in scores]) for m in scores[0]}, "query_count": len(scores)})
    for qi, q in enumerate(queries):
        if len(examples) < 50 and fold == 1:
            examples.append({"fold": fold, "role_group": q["role_group"], "exp_level": q["exp_level"],
                "known_skills": json.dumps(sorted(q["known"]), ensure_ascii=False),
                "hidden_target_skills": json.dumps(sorted(q["target"]), ensure_ascii=False),
                "apriori_recommendations": json.dumps(predictions["Apriori"][qi], ensure_ascii=False),
                "apriori_exp_recommendations": json.dumps(predictions["Apriori + EXP"][qi], ensure_ascii=False)})

fold_quality_df = pd.DataFrame(fold_quality)
quality_df = fold_quality_df.groupby(["Model", "K"])[metric_names].agg(["mean", "std"]).reset_index()
quality_df.columns = ["Model", "K", *[f"{m}_{s}" for m in metric_names for s in ["mean", "std"]]]
quality_df["Model"] = pd.Categorical(quality_df["Model"], MODELS, ordered=True)
quality_df = quality_df.sort_values(["K", "Model"]).reset_index(drop=True)
quality_df["Model"] = quality_df["Model"].astype(str)
quality_df["folds"] = N_SPLITS * N_REPEATS
bench_folds = pd.DataFrame(fold_benchmarks)
benchmark_df = bench_folds.groupby("Model").agg(
    **{"Runtime median (s)": ("Runtime (s)", "median"), "Runtime std (s)": ("Runtime (s)", "std"),
       "Number of Rules mean": ("Number of Rules", "mean"), "Number of Rules std": ("Number of Rules", "std"),
       "Memory Usage median (MB)": ("Memory Usage (MB)", "median"),
       "Memory Usage std (MB)": ("Memory Usage (MB)", "std")}).reindex(MODELS).reset_index()
role_quality_df = pd.DataFrame(fold_role_quality)
display(quality_df.rename(columns={"Model": "Mô hình"}).round(4)); display(benchmark_df.rename(columns={"Model": "Mô hình", "Runtime median (s)": "Runtime trung vị (giây)",
    "Runtime std (s)": "Độ lệch chuẩn runtime", "Number of Rules mean": "Số luật trung bình",
    "Number of Rules std": "Độ lệch chuẩn số luật", "Memory Usage median (MB)": "Bộ nhớ Python trung vị (MB)",
    "Memory Usage std (MB)": "Độ lệch chuẩn bộ nhớ"}).round(4))

### 3.1 So sánh cặp theo từng fold

`Δ = mô hình đầu − mô hình đối chứng` tính theo từng fold; *Fold tốt hơn* đếm số fold mà mô hình đầu thắng. Kiểm định Wilcoxon chỉ mang tính tham khảo vì các fold lặp lại dùng chung dữ liệu nên không độc lập và p-value có xu hướng quá lạc quan. Một cải thiện đáng tin khi `Δ` cùng dấu ở hầu hết các fold.

In [ ]:
PAIRS = [("Apriori + EXP", "Apriori"), ("FP-Growth + EXP", "FP-Growth"),
         ("Phổ biến theo role + EXP", "Phổ biến theo role"),
         ("Apriori + EXP", "Phổ biến theo role + EXP"), ("Apriori + EXP", "Phổ biến theo role")]
pair_rows = []
for name, ref in PAIRS:
    for k in TOP_K_VALUES:
        a = fold_quality_df[(fold_quality_df.Model == name) & (fold_quality_df.K == k)].set_index("fold")
        b = fold_quality_df[(fold_quality_df.Model == ref) & (fold_quality_df.K == k)].set_index("fold")
        for metric in ["Precision@K", "Recall@K", "Hit@K", "NDCG@K"]:
            d = (a[metric] - b[metric]).dropna()
            try: p_value = wilcoxon(d).pvalue if (d != 0).any() else 1.0
            except ValueError: p_value = np.nan
            pair_rows.append({"So sánh": f"{name} − {ref}", "K": k, "Metric": metric,
                "Δ trung bình": d.mean(), "Δ std": d.std(), "Fold tốt hơn": int((d > 0).sum()),
                "Fold kém hơn": int((d < 0).sum()), "Tổng fold": len(d), "p (Wilcoxon, tham khảo)": p_value})
paired_df = pd.DataFrame(pair_rows)
display(paired_df.round(4))

### 3.2 Kết quả theo từng role

NDCG@5 trung bình theo role. Điểm toàn cục có thể bị các role lớn chi phối, nên cần xem role nào thực sự hưởng lợi từ thông tin cấp bậc.

In [ ]:
by_role = role_quality_df[role_quality_df.K == 5].groupby(["role_group", "Model"])["NDCG@K"].mean().unstack("Model")[MODELS]
by_role["Δ Phổ biến + EXP − Phổ biến"] = by_role["Phổ biến theo role + EXP"] - by_role["Phổ biến theo role"]
by_role["Δ Apriori + EXP − Apriori"] = by_role["Apriori + EXP"] - by_role["Apriori"]
by_role.insert(0, "Số truy vấn/fold", (role_quality_df[(role_quality_df.K == 5) & (role_quality_df.Model == "Apriori")]
                                      .groupby("role_group")["query_count"].mean().round(1)))
display(by_role.round(4))

## 4. Luật cuối cùng và độ bao phủ theo role

Các bảng luật cuối được khai phá từ toàn bộ tin có kỹ năng để tiện rà soát; điểm chất lượng phía trên được tính trên các fold test. Bảng theo role giúp thấy số luật và phạm vi gợi ý của từng nhóm, và bao nhiêu luật của biến thể + EXP thực sự dùng item cấp bậc.

In [ ]:
rules_apriori = mine_context_rules(jobs, transactions, "Apriori")
rules_fpgrowth = mine_context_rules(jobs, transactions, "FP-Growth")
rules_apriori_exp = mine_context_rules(jobs, transactions, "Apriori", use_exp=True)
rules_fpgrowth_exp = mine_context_rules(jobs, transactions, "FP-Growth", use_exp=True)
for rules in (rules_apriori_exp, rules_fpgrowth_exp):
    rules["uses_exp"] = rules.known_skills.map(lambda xs: any(x.startswith("EXP_") for x in xs))
coverage_rows = []
for role in sorted(jobs.role_group.unique()):
    a, f = rules_apriori[rules_apriori.role_group.eq(role)], rules_fpgrowth[rules_fpgrowth.role_group.eq(role)]
    ae, fe = rules_apriori_exp[rules_apriori_exp.role_group.eq(role)], rules_fpgrowth_exp[rules_fpgrowth_exp.role_group.eq(role)]
    coverage_rows.append({"role_group": role, "query_jobs": int((jobs.role_group.eq(role) & jobs.query_eligible).sum()),
        "apriori_rules": len(a), "apriori_next_skills": a.next_skill.nunique(),
        "fpgrowth_rules": len(f), "fpgrowth_next_skills": f.next_skill.nunique(),
        "apriori_exp_rules": len(ae), "apriori_exp_rules_using_exp": int(ae.uses_exp.sum()),
        "apriori_exp_next_skills": ae.next_skill.nunique(),
        "fpgrowth_exp_rules": len(fe), "fpgrowth_exp_next_skills": fe.next_skill.nunique()})
role_coverage_df = pd.DataFrame(coverage_rows)
display(role_coverage_df.rename(columns={"role_group": "Nhóm role"}))

def export_rules(rules, path):
    output = rules.copy()
    if not output.empty:
        output["known_skills"] = output.known_skills.map(lambda xs: json.dumps(sorted(xs), ensure_ascii=False))
        output["rule_length_including_role"] = output.known_skills.map(lambda x: len(json.loads(x)) + 2)
    output.to_csv(path, index=False, encoding="utf-8-sig")

### 4.1 Ví dụ luật có dùng cấp bậc

Hai luật có lift cao nhất mỗi role mà vế trái chứa item `EXP_*` (tối đa 30 dòng). Luật dạng `{EXP_Senior, skill}` cho thấy skill nào được kỳ vọng thêm khi tin đòi hỏi nhiều kinh nghiệm hơn.

In [ ]:
exp_rules = rules_apriori_exp[rules_apriori_exp.uses_exp].copy()
exp_rules["known_skills"] = exp_rules.known_skills.map(lambda xs: " + ".join(sorted(xs)))
display(exp_rules.sort_values(["role_group", "lift"], ascending=[True, False]).groupby("role_group").head(2)
        [["role_group", "known_skills", "next_skill", "support_count", "confidence", "lift"]].head(30).round(3)
        .reset_index(drop=True))

## 5. Biểu đồ và file kết quả

Các bảng CSV và biểu đồ được lưu trong `result/problem_2/`. Bộ nhớ là lượng cấp phát đỉnh của Python đo bằng `tracemalloc`, không phải toàn bộ bộ nhớ native của NumPy/pandas.

In [ ]:
PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974", "#64B5CD"]
plot_df = quality_df.copy()
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
for ax, metric, vn_title in zip(axes, ["Precision@K", "Recall@K", "Hit@K", "NDCG@K"],
    ["Precision@K — Độ chính xác", "Recall@K — Độ bao phủ", "Hit@K — Tỷ lệ trúng", "NDCG@K — Thứ hạng"]):
    plot_df.pivot(index="Model", columns="K", values=f"{metric}_mean").reindex(MODELS).plot(
        kind="bar", ax=ax, ylim=(0, 1), rot=35)
    ax.set_title(vn_title); ax.set_ylabel(f"Trung bình qua {N_SPLITS * N_REPEATS} lần đánh giá"); ax.set_xlabel("")
    ax.grid(axis="y", linestyle="--", alpha=.35); ax.legend(title="K")
    for label in ax.get_xticklabels(): label.set_ha("right")
fig.suptitle("Bài toán 2 — Gợi ý kỹ năng theo ngữ cảnh (có và không có cấp bậc kinh nghiệm)"); fig.tight_layout()
fig.savefig(FIGURES_DIR / "recommendation_quality.png", dpi=180, bbox_inches="tight"); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, col, title in zip(axes, ["Runtime median (s)", "Number of Rules mean", "Memory Usage median (MB)"],
                          ["Thời gian chạy", "Số luật trung bình", "Bộ nhớ Python cực đại"]):
    ax.bar(benchmark_df.Model, benchmark_df[col], color=PALETTE)
    ax.set_title(title); ax.tick_params(axis="x", rotation=35); ax.grid(axis="y", linestyle="--", alpha=.35)
    for label in ax.get_xticklabels(): label.set_ha("right")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "benchmark.png", dpi=180, bbox_inches="tight"); plt.show()
export_rules(rules_apriori, RESULT_DIR / "rules_apriori.csv")
export_rules(rules_fpgrowth, RESULT_DIR / "rules_fpgrowth.csv")
export_rules(rules_apriori_exp, RESULT_DIR / "rules_apriori_exp.csv")
export_rules(rules_fpgrowth_exp, RESULT_DIR / "rules_fpgrowth_exp.csv")
quality_df.to_csv(RESULT_DIR / "quality_metrics.csv", index=False, encoding="utf-8-sig")
fold_quality_df.to_csv(RESULT_DIR / "quality_metrics_by_fold.csv", index=False, encoding="utf-8-sig")
paired_df.to_csv(RESULT_DIR / "paired_comparison.csv", index=False, encoding="utf-8-sig")
benchmark_df.to_csv(RESULT_DIR / "benchmark.csv", index=False, encoding="utf-8-sig")
role_coverage_df.to_csv(RESULT_DIR / "role_coverage.csv", index=False, encoding="utf-8-sig")
role_quality_df.to_csv(RESULT_DIR / "quality_by_role.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(examples).to_csv(RESULT_DIR / "recommendation_examples.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả tại:", RESULT_DIR.resolve())

## 6. Diễn giải

Cách đọc kết quả, theo thứ tự:

1. **Luật + EXP so với luật gốc**: vì tập luật + EXP chứa toàn bộ luật gốc, `Δ` dương ở hầu hết các fold nghĩa là item cấp bậc cung cấp thêm thông tin hữu ích cho ngữ cảnh skill đã biết.
2. **Luật + EXP so với baseline tần suất có EXP**: cho biết cơ chế luật theo ngữ cảnh có thêm giá trị gì ngoài chính thông tin kinh nghiệm. Nếu baseline này vẫn thắng ở K=10, nguyên nhân thường là độ bao phủ của luật thấp (luật không trả đủ K gợi ý), nên cần đọc cùng cột `Recommendation coverage`.
3. **Theo role** (mục 3.2): chỉ các role có đủ tin ở nhiều cấp bậc mới hưởng lợi rõ rệt; với role nhỏ, `Δ` gần 0 là điều dự kiến.
4. **Giới hạn của thí nghiệm**: kết quả đo khả năng khôi phục skill bị ẩn khỏi hồ sơ tuyển dụng, không chứng minh một lộ trình học theo thứ tự thời gian. Nhãn `Intern` nghĩa là "không yêu cầu kinh nghiệm", và các tin Intern thường liệt kê rất ít skill nên truy vấn từ nhóm này nghèo ngữ cảnh.